# 🌙 Lunara — Feature Lifecycle Intelligence System

**UMBC Data Science Master's Degree Capstone — Dr. Chaojie (Jay) Wang**

**Author:** Venkata Sai Chandravadan Sobila

---

## Project Overview

This notebook implements an end-to-end Feature Lifecycle Intelligence System for a fictional e-commerce platform called **Lunara**. The system:

1. **Generates synthetic data** (proxies for Google Analytics, GitHub Issues, NASA Defect datasets)
2. **ETL Pipeline** — Extracts, transforms, and aggregates data into weekly feature-level metrics
3. **Exploratory Data Analysis** — Comprehensive visualizations with Plotly
4. **Adoption Forecasting** — LightGBM/Gradient Boosting to predict future weekly active users
5. **Feature Value Scoring** — Transparent composite 0-100 score
6. **Lifecycle Classification** — Invest / Maintain / Refactor / Sunset
7. **RAG Explainability** — Evidence-grounded decision reports
8. **Interactive Dashboard** — Streamlit app

### Lunara Features Tracked
| Feature | Type | Description |
|---------|------|-------------|
| Search | Discovery | Product search |
| Recommendations | Discovery | Personalized recommendations |
| Wishlist | Engagement | Save items for later |
| Reviews | Engagement | Product reviews & ratings |
| Notifications | Retention | Push/email notifications |
| Checkout | Conversion | Purchase & payment flow |

---
## 0. Setup & Installation

Run this cell first to install all required packages.

In [61]:
# INSTALL DEPENDENCIES
!pip install -q lightgbm plotly scikit-learn pandas numpy faiss-cpu sentence-transformers

print("\n All packages installed!")


 All packages installed!


In [62]:
# IMPORTS
import numpy as np
import pandas as pd
import os
import json
import pickle
import warnings
from datetime import datetime, timedelta
from pathlib import Path

# ML
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_percentage_error,
    accuracy_score, f1_score, classification_report, confusion_matrix
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
import joblib

# Try LightGBM (preferred), fallback to sklearn
try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
    print(" LightGBM available")
except ImportError:
    HAS_LIGHTGBM = False
    print(" LightGBM not found, using sklearn GradientBoosting")

# Visualization
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

# RAG
try:
    from sentence_transformers import SentenceTransformer
    HAS_SBERT = True
    print(" Sentence-Transformers available")
except ImportError:
    HAS_SBERT = False
    print(" Sentence-Transformers not found, using TF-IDF for RAG")

try:
    import faiss
    HAS_FAISS = True
    print(" FAISS available")
except ImportError:
    HAS_FAISS = False
    print(" FAISS not found, using brute-force retrieval")

warnings.filterwarnings('ignore')
np.random.seed(42)

# Create directories
for d in ['data/raw', 'data/processed', 'models/artifacts', 'rag/index']:
    os.makedirs(d, exist_ok=True)

print("\n All imports successful!")

 LightGBM available
 Sentence-Transformers available
 FAISS available

 All imports successful!


---
## 1. Configuration

Central configuration for all features, weights, and hyperparameters.

In [63]:
# CONFIGURATION

# Lunara Feature Definitions
FEATURES = ["Search", "Recommendations", "Wishlist", "Reviews", "Notifications", "Checkout"]

FEATURE_TYPES = {
    "Search": "discovery",
    "Recommendations": "discovery",
    "Wishlist": "engagement",
    "Reviews": "engagement",
    "Notifications": "retention",
    "Checkout": "conversion",
}

# Data generation parameters
NUM_USERS = 8000
NUM_SESSIONS = 50000
NUM_ISSUES = 5000
NUM_MODULES = 2000
DATE_START = "2023-01-01"
DATE_END = "2024-12-31"

# Value Score Weights
SCORE_WEIGHTS = {
    "adoption_trend": 0.30,
    "engagement_intensity": 0.25,
    "issue_burden": -0.25,
    "defect_risk": -0.20,
}

# Lifecycle Thresholds
LIFECYCLE_THRESHOLDS = {
    "Invest": (75, 100),
    "Maintain": (50, 75),
    "Refactor": (25, 50),
    "Sunset": (0, 25),
}

LIFECYCLE_LABELS = ["Invest", "Maintain", "Refactor", "Sunset"]

# Forecasting config
FORECAST_HORIZON = 12
LAG_FEATURES = [1, 2, 3, 4, 8, 12]
ROLLING_WINDOWS = [4, 8, 12]

# Feature behavior profiles
PROFILES = {
    "Search": {
        "base_popularity": 0.28, "trend": "growing",
        "avg_pageviews": 4.5, "txn_rate": 0.08,
        "issue_rate": 0.04, "defect_prob": 0.12, "complexity_mean": 22,
    },
    "Recommendations": {
        "base_popularity": 0.18, "trend": "growing",
        "avg_pageviews": 3.8, "txn_rate": 0.12,
        "issue_rate": 0.06, "defect_prob": 0.18, "complexity_mean": 35,
    },
    "Wishlist": {
        "base_popularity": 0.12, "trend": "stable",
        "avg_pageviews": 2.5, "txn_rate": 0.03,
        "issue_rate": 0.08, "defect_prob": 0.15, "complexity_mean": 18,
    },
    "Reviews": {
        "base_popularity": 0.15, "trend": "declining",
        "avg_pageviews": 3.0, "txn_rate": 0.02,
        "issue_rate": 0.10, "defect_prob": 0.25, "complexity_mean": 28,
    },
    "Notifications": {
        "base_popularity": 0.10, "trend": "declining",
        "avg_pageviews": 1.8, "txn_rate": 0.01,
        "issue_rate": 0.15, "defect_prob": 0.30, "complexity_mean": 40,
    },
    "Checkout": {
        "base_popularity": 0.17, "trend": "growing",
        "avg_pageviews": 5.2, "txn_rate": 0.45,
        "issue_rate": 0.05, "defect_prob": 0.10, "complexity_mean": 30,
    },
}

print(" Configuration loaded")
print(f"  Features: {FEATURES}")
print(f"  Score weights: {SCORE_WEIGHTS}")

 Configuration loaded
  Features: ['Search', 'Recommendations', 'Wishlist', 'Reviews', 'Notifications', 'Checkout']
  Score weights: {'adoption_trend': 0.3, 'engagement_intensity': 0.25, 'issue_burden': -0.25, 'defect_risk': -0.2}


---
## 2. Data Generation

- **Google Analytics** → User behavior sessions
- **GitHub Issues** → Operational issues
- **NASA Defect Data** → Software quality metrics

In [64]:
# DATA GENERATION
def _trend_multiplier(trend, week_idx, total_weeks):
    """Apply realistic trend to popularity over time."""
    progress = week_idx / total_weeks
    if trend == "growing":
        return 1.0 + 0.6 * progress + 0.1 * np.sin(progress * 4 * np.pi)
    elif trend == "declining":
        return 1.0 - 0.4 * progress + 0.05 * np.sin(progress * 3 * np.pi)
    else:
        return 1.0 + 0.08 * np.sin(progress * 6 * np.pi)


def generate_user_behavior():
    """Generate Google Analytics-style session data."""
    print("Generating data for lunara user behavior data...")
    start = pd.Timestamp(DATE_START)
    end = pd.Timestamp(DATE_END)
    total_days = (end - start).days
    total_weeks = total_days // 7

    devices = ["desktop", "mobile", "tablet"]
    device_weights = [0.45, 0.42, 0.13]
    sources = ["organic", "paid", "direct", "social", "email", "referral"]
    source_weights = [0.32, 0.20, 0.18, 0.15, 0.08, 0.07]
    countries = ["US", "UK", "IN", "DE", "CA", "FR", "AU", "BR", "JP", "MX"]
    user_ids = [f"U{str(i).zfill(6)}" for i in range(NUM_USERS)]

    records = []
    for i in range(NUM_SESSIONS):
        day_offset = np.random.randint(0, total_days)
        session_date = start + timedelta(days=int(day_offset))
        week_idx = day_offset // 7

        probs = []
        for f in FEATURES:
            p = PROFILES[f]["base_popularity"]
            p *= _trend_multiplier(PROFILES[f]["trend"], week_idx, total_weeks)
            probs.append(max(p, 0.01))
        total_p = sum(probs)
        probs = [p / total_p for p in probs]

        feature = np.random.choice(FEATURES, p=probs)
        profile = PROFILES[feature]
        pageviews = max(1, int(np.random.poisson(profile["avg_pageviews"])))
        has_transaction = 1 if np.random.random() < profile["txn_rate"] else 0
        revenue = round(np.random.lognormal(3.5, 1.0), 2) if has_transaction else 0.0
        session_duration = max(10, int(np.random.exponential(180) * (pageviews / 3)))
        bounce = 1 if pageviews == 1 and np.random.random() < 0.6 else 0

        records.append({
            "session_id": f"S{str(i).zfill(8)}",
            "user_id": np.random.choice(user_ids),
            "session_date": session_date.strftime("%Y-%m-%d"),
            "feature": feature,
            "pageviews": pageviews,
            "session_duration_sec": session_duration,
            "transactions": has_transaction,
            "revenue": revenue,
            "bounces": bounce,
            "device": np.random.choice(devices, p=device_weights),
            "traffic_source": np.random.choice(sources, p=source_weights),
            "country": np.random.choice(countries),
        })

    df = pd.DataFrame(records)
    df.to_csv("data/raw/user_behavior.csv", index=False)
    print(f"  → {len(df)} sessions saved")
    return df


def generate_operational_issues():
    """Generate GitHub-style issue data."""
    print("Generating data for lunara operational issues...")
    start = pd.Timestamp(DATE_START)
    end = pd.Timestamp(DATE_END)
    total_days = (end - start).days
    total_weeks = total_days // 7

    issue_types = ["bug", "feature_request", "performance", "outage", "ux_complaint"]
    severities = ["low", "medium", "high", "critical"]
    states = ["open", "closed"]

    records = []
    for i in range(NUM_ISSUES):
        day_offset = np.random.randint(0, total_days)
        created = start + timedelta(days=int(day_offset))
        week_idx = day_offset // 7

        probs = [PROFILES[f]["issue_rate"] for f in FEATURES]
        for idx, f in enumerate(FEATURES):
            if PROFILES[f]["trend"] == "declining":
                probs[idx] *= (1 + 0.5 * week_idx / total_weeks)
        total_p = sum(probs)
        probs = [p / total_p for p in probs]

        feature = np.random.choice(FEATURES, p=probs)
        issue_type = np.random.choice(issue_types, p=[0.35, 0.20, 0.20, 0.10, 0.15])

        if issue_type == "outage":
            severity = np.random.choice(severities, p=[0.05, 0.15, 0.40, 0.40])
        elif issue_type == "bug":
            severity = np.random.choice(severities, p=[0.15, 0.35, 0.35, 0.15])
        else:
            severity = np.random.choice(severities, p=[0.30, 0.40, 0.20, 0.10])

        state = np.random.choice(states, p=[0.30, 0.70])
        resolution_days = int(np.random.exponential(7)) if state == "closed" else None

        records.append({
            "issue_id": f"I{str(i).zfill(6)}",
            "feature": feature,
            "created_at": created.strftime("%Y-%m-%d"),
            "issue_type": issue_type,
            "severity": severity,
            "state": state,
            "resolution_days": resolution_days,
            "title": f"[{feature}] {issue_type.replace('_', ' ').title()} #{i}",
        })

    df = pd.DataFrame(records)
    df.to_csv("data/raw/operational_issues.csv", index=False)
    print(f"  → {len(df)} issues saved")
    return df


def generate_software_quality():
    """Generate NASA-style software defect data."""
    print("Generating data for lunara software quality data...")
    records = []
    modules_per_feature = NUM_MODULES // len(FEATURES)

    for feature in FEATURES:
        profile = PROFILES[feature]
        for j in range(modules_per_feature):
            loc = max(10, int(np.random.lognormal(np.log(profile["complexity_mean"] * 10), 0.6)))
            cyclomatic = max(1, int(np.random.poisson(profile["complexity_mean"])))
            halstead_volume = round(loc * np.random.uniform(3, 8), 1)
            halstead_effort = round(halstead_volume * np.random.uniform(5, 20), 1)
            essential_complexity = max(1, int(cyclomatic * np.random.uniform(0.3, 0.7)))
            design_complexity = max(1, int(cyclomatic * np.random.uniform(0.5, 0.9)))

            defect_score = (
                0.002 * cyclomatic + 0.0001 * loc
                + 0.00005 * halstead_effort + profile["defect_prob"] * 0.5
            )
            has_defect = 1 if np.random.random() < min(defect_score, 0.9) else 0

            records.append({
                "module_id": f"M_{feature[:3].upper()}_{str(j).zfill(4)}",
                "feature": feature,
                "loc": loc,
                "cyclomatic_complexity": cyclomatic,
                "essential_complexity": essential_complexity,
                "design_complexity": design_complexity,
                "halstead_volume": halstead_volume,
                "halstead_effort": halstead_effort,
                "num_operators": max(5, int(loc * np.random.uniform(0.1, 0.3))),
                "num_operands": max(5, int(loc * np.random.uniform(0.15, 0.35))),
                "has_defect": has_defect,
            })

    df = pd.DataFrame(records)
    df.to_csv("data/raw/software_quality.csv", index=False)
    print(f"  → {len(df)} modules saved")
    return df


# GENERATE ALL DATA
print("LUNARA — Data Generation")
raw_behavior = generate_user_behavior()
raw_issues = generate_operational_issues()
raw_quality = generate_software_quality()
print("\n All datasets generated!")

LUNARA — Data Generation
Generating data for lunara user behavior data...
  → 50000 sessions saved
Generating data for lunara operational issues...
  → 5000 issues saved
Generating data for lunara software quality data...
  → 1998 modules saved

 All datasets generated!


In [65]:
# Quick look at the generated data
print("User Behavior")
display(raw_behavior.head())
print(f"\nShape: {raw_behavior.shape}")
print(f"Features distribution:\n{raw_behavior['feature'].value_counts()}")

print("\n Operational Issues")
display(raw_issues.head())

print("\n Software Quality")
display(raw_quality.head())

User Behavior


,session_id,user_id,session_date,feature,pageviews,session_duration_sec,transactions,revenue,bounces,device,traffic_source,country
0,S00000000,U005578,2023-04-13,Notifications,1,35,0,0.0,1,mobile,direct,BR
1,S00000001,U004843,2023-05-11,Search,4,178,0,0.0,0,desktop,direct,MX
2,S00000002,U003556,2024-04-20,Checkout,3,113,0,0.0,0,mobile,organic,AU
3,S00000003,U006349,2023-10-01,Checkout,6,12,0,0.0,0,desktop,organic,DE
4,S00000004,U004297,2023-01-02,Recommendations,2,71,0,0.0,0,tablet,direct,BR



Shape: (50000, 12)
Features distribution:
feature
Search             15984
Recommendations    10070
Checkout            9682
Wishlist            5323
Reviews             5297
Notifications       3644
Name: count, dtype: int64

 Operational Issues


,issue_id,feature,created_at,issue_type,severity,state,resolution_days,title
0,I000000,Wishlist,2023-05-10,bug,critical,closed,7.0,[Wishlist] Bug #0
1,I000001,Reviews,2024-03-01,outage,medium,closed,2.0,[Reviews] Outage #1
2,I000002,Reviews,2024-04-01,ux_complaint,low,closed,4.0,[Reviews] Ux Complaint #2
3,I000003,Checkout,2023-06-20,bug,medium,closed,6.0,[Checkout] Bug #3
4,I000004,Wishlist,2023-03-14,performance,low,closed,12.0,[Wishlist] Performance #4



 Software Quality


,module_id,feature,loc,cyclomatic_complexity,essential_complexity,design_complexity,halstead_volume,halstead_effort,num_operators,num_operands,has_defect
0,M_SEA_0000,Search,144,21,11,17,1127.5,7034.5,34,26,0
1,M_SEA_0001,Search,81,23,12,17,363.3,3194.6,15,20,0
2,M_SEA_0002,Search,158,22,13,18,1154.4,10597.0,43,47,1
3,M_SEA_0003,Search,802,18,9,14,4604.2,89703.5,199,241,1
4,M_SEA_0004,Search,594,21,12,14,3569.0,27187.5,161,192,1


---
## 3. ETL Pipeline

Extract, transform, and aggregate raw data into a weekly feature-level derived dataset.

**Granularity:** One row = one feature × one week

In [66]:
# ETL PIPELINE
def _min_max_scale(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)


# Load raw data with proper types
behavior_df = pd.read_csv("data/raw/user_behavior.csv", parse_dates=["session_date"])
issues_df = pd.read_csv("data/raw/operational_issues.csv", parse_dates=["created_at"])
quality_df = pd.read_csv("data/raw/software_quality.csv")

print("[1/4] Aggregating user behavior...")
behavior_df["week"] = behavior_df["session_date"].dt.to_period("W").apply(lambda x: x.start_time)
weekly_behavior = behavior_df.groupby(["feature", "week"]).agg(
    weekly_active_users=("user_id", "nunique"),
    total_sessions=("session_id", "count"),
    avg_pageviews=("pageviews", "mean"),
    avg_session_duration=("session_duration_sec", "mean"),
    total_transactions=("transactions", "sum"),
    total_revenue=("revenue", "sum"),
    bounce_rate=("bounces", "mean"),
).reset_index()
weekly_behavior = weekly_behavior.round({"avg_pageviews": 2, "avg_session_duration": 1, "bounce_rate": 4, "total_revenue": 2})

print("[2/4] Aggregating operational issues...")
issues_df["week"] = issues_df["created_at"].dt.to_period("W").apply(lambda x: x.start_time)
weekly_issues = issues_df.groupby(["feature", "week"]).agg(
    issue_count=("issue_id", "count"),
    critical_issues=("severity", lambda x: (x == "critical").sum()),
    high_issues=("severity", lambda x: (x == "high").sum()),
    open_issues=("state", lambda x: (x == "open").sum()),
    avg_resolution_days=("resolution_days", "mean"),
).reset_index()

print("[3/4] Computing software quality metrics...")
quality_summary = quality_df.groupby("feature").agg(
    avg_cyclomatic=("cyclomatic_complexity", "mean"),
    avg_halstead_effort=("halstead_effort", "mean"),
    avg_loc=("loc", "mean"),
    defect_rate=("has_defect", "mean"),
    total_modules=("module_id", "count"),
    defective_modules=("has_defect", "sum"),
).reset_index()
quality_summary["defect_risk_score"] = (
    0.4 * _min_max_scale(quality_summary["defect_rate"])
    + 0.3 * _min_max_scale(quality_summary["avg_cyclomatic"])
    + 0.3 * _min_max_scale(quality_summary["avg_halstead_effort"])
).round(4)

print("[4/4] Merging into derived dataset...")
derived = weekly_behavior.merge(weekly_issues, on=["feature", "week"], how="left")
for col in ["issue_count", "critical_issues", "high_issues", "open_issues", "avg_resolution_days"]:
    derived[col] = derived[col].fillna(0)

quality_cols = ["feature", "defect_risk_score", "avg_cyclomatic", "avg_halstead_effort", "defect_rate"]
derived = derived.merge(quality_summary[quality_cols], on="feature", how="left")
derived["feature_type"] = derived["feature"].map(FEATURE_TYPES)
derived["week_number"] = derived.groupby("feature").cumcount() + 1
derived = derived.sort_values(["feature", "week"]).reset_index(drop=True)

# Save
derived.to_csv("data/processed/weekly_feature_data.csv", index=False)

print(f"\n Derived dataset: {derived.shape}")
print(f"  Features: {derived['feature'].nunique()}")
print(f"  Weeks: {derived['week'].nunique()}")
display(derived.head(10))

[1/4] Aggregating user behavior...
[2/4] Aggregating operational issues...
[3/4] Computing software quality metrics...
[4/4] Merging into derived dataset...

 Derived dataset: (636, 20)
  Features: 6
  Weeks: 106


,feature,week,weekly_active_users,total_sessions,avg_pageviews,avg_session_duration,total_transactions,total_revenue,bounce_rate,issue_count,critical_issues,high_issues,open_issues,avg_resolution_days,defect_risk_score,avg_cyclomatic,avg_halstead_effort,defect_rate,feature_type,week_number
0,Checkout,2022-12-26,17,17,5.06,327.6,8,741.43,0.0000,0.0,0.0,0.0,0.0,0.000000,0.6042,30.147147,24660.881682,0.783784,conversion,1
1,Checkout,2023-01-02,110,110,5.25,296.1,51,3286.28,0.0364,5.0,1.0,0.0,1.0,6.000000,0.6042,30.147147,24660.881682,0.783784,conversion,2
2,Checkout,2023-01-09,67,67,5.25,290.9,30,1771.89,0.0149,5.0,0.0,2.0,0.0,6.400000,0.6042,30.147147,24660.881682,0.783784,conversion,3
3,Checkout,2023-01-16,76,76,4.64,249.6,41,1753.79,0.0132,3.0,0.0,3.0,2.0,30.000000,0.6042,30.147147,24660.881682,0.783784,conversion,4
4,Checkout,2023-01-23,97,97,4.90,294.1,41,2492.85,0.0103,3.0,1.0,1.0,0.0,3.666667,0.6042,30.147147,24660.881682,0.783784,conversion,5
5,Checkout,2023-01-30,88,88,4.94,268.1,37,1860.24,0.0114,6.0,0.0,3.0,1.0,4.000000,0.6042,30.147147,24660.881682,0.783784,conversion,6
6,Checkout,2023-02-06,71,72,5.32,355.5,31,1629.89,0.0139,3.0,0.0,0.0,2.0,1.000000,0.6042,30.147147,24660.881682,0.783784,conversion,7
7,Checkout,2023-02-13,84,84,4.96,341.4,33,1386.48,0.0119,2.0,1.0,0.0,1.0,8.000000,0.6042,30.147147,24660.881682,0.783784,conversion,8
8,Checkout,2023-02-20,84,84,5.29,291.3,29,2081.72,0.0000,6.0,0.0,3.0,0.0,3.500000,0.6042,30.147147,24660.881682,0.783784,conversion,9
9,Checkout,2023-02-27,91,91,4.73,315.4,41,1907.65,0.0110,3.0,2.0,0.0,0.0,6.000000,0.6042,30.147147,24660.881682,0.783784,conversion,10


---
## 4. Exploratory Data Analysis (EDA)

Comprehensive visualizations of feature behavior, engagement, issues, and quality.

In [67]:
# EDA: Summary Statistics
print("SUMMARY STATISTICS")
print("=" * 60)

summary = derived.groupby("feature").agg({
    "weekly_active_users": ["mean", "std", "min", "max"],
    "total_sessions": ["mean", "sum"],
    "total_revenue": ["sum", "mean"],
    "issue_count": ["sum", "mean"],
    "defect_risk_score": "first",
}).round(2)

display(summary)

print(f"\nData Quality:")
print(f"  Missing values: {derived.isnull().sum().sum()}")
print(f"  Duplicate rows: {derived.duplicated().sum()}")
print(f"  Date range: {derived['week'].min()} to {derived['week'].max()}")

SUMMARY STATISTICS


weekly_active_users                 total_sessions         \
                               mean    std min  max           mean    sum   
feature                                                                     
Checkout                      90.78  14.92  15  112          91.34   9682   
Notifications                 34.31   9.84   1   69          34.38   3644   
Recommendations               94.43  15.50  13  124          95.00  10070   
Reviews                       49.83  13.49   3   81          49.97   5297   
Search                       149.36  23.60  15  195         150.79  15984   
Wishlist                      50.04  11.52   4   74          50.22   5323   

                total_revenue          issue_count        defect_risk_score  
                          sum     mean         sum   mean             first  
feature                                                                      
Checkout            233449.50  2202.35       500.0   4.72              0.60  
Notifications         1632.95    15.41      1694.0  15.98              1.00  
Recommendations      68267.12   644.03       525.0   4.95              0.74  
Reviews               4638.13    43.76      1156.0  10.91              0.64  
Search               71666.95   676.10       367.0   3.46              0.24  
Wishlist             10832.75   102.20       758.0   7.15              0.00


Data Quality:
  Missing values: 0
  Duplicate rows: 0
  Date range: 2022-12-26 00:00:00 to 2024-12-30 00:00:00


In [68]:
# EDA: Weekly Active Users Trends
fig = px.line(
    derived, x="week", y="weekly_active_users",
    color="feature",
    title="Weekly Active Users by Feature Over Time",
    labels={"weekly_active_users": "Weekly Active Users", "week": "Week"},
    template="plotly_white",
)
fig.update_layout(width=1000, height=500, legend=dict(orientation="h", y=1.08))
fig.show()

In [69]:
# EDA: Engagement Heatmap
metrics = derived.groupby("feature").agg({
    "avg_pageviews": "mean",
    "avg_session_duration": "mean",
    "bounce_rate": "mean",
    "total_transactions": "sum",
    "total_revenue": "sum",
}).round(2)

normalized = (metrics - metrics.min()) / (metrics.max() - metrics.min())

fig = go.Figure(data=go.Heatmap(
    z=normalized.values,
    x=normalized.columns.tolist(),
    y=normalized.index.tolist(),
    colorscale="RdYlGn",
    text=metrics.values.round(2),
    texttemplate="%{text}",
    textfont={"size": 11},
))
fig.update_layout(title="Feature Engagement Heatmap", width=800, height=400, template="plotly_white")
fig.show()

In [70]:
# EDA: Issue Burden Over Time

issue_weekly = derived.groupby(["week", "feature"])["issue_count"].sum().reset_index()

fig = px.area(
    issue_weekly, x="week", y="issue_count",
    color="feature",
    title="Operational Issue Burden Over Time",
    labels={"issue_count": "Issues per Week"},
    template="plotly_white",
)
fig.update_layout(width=1000, height=500)
fig.show()

In [71]:
# EDA: Defect Distribution by Feature
fig = px.box(
    raw_quality, x="feature", y="cyclomatic_complexity",
    color="has_defect",
    title="Cyclomatic Complexity by Feature & Defect Status",
    template="plotly_white",
    color_discrete_map={0: "#2ecc71", 1: "#e74c3c"},
)
fig.update_layout(width=900, height=500)
fig.show()

In [72]:
# EDA: Correlation Matrix

numeric_cols = [
    "weekly_active_users", "total_sessions", "avg_pageviews",
    "avg_session_duration", "total_revenue", "bounce_rate",
    "issue_count", "critical_issues", "defect_risk_score",
]
corr = derived[numeric_cols].corr().round(3)

fig = go.Figure(data=go.Heatmap(
    z=corr.values, x=corr.columns.tolist(), y=corr.index.tolist(),
    colorscale="RdBu_r", zmid=0,
    text=corr.values.round(2), texttemplate="%{text}", textfont={"size": 9},
))
fig.update_layout(title="Feature Metric Correlation Matrix", width=800, height=700, template="plotly_white")
fig.show()

In [73]:
# EDA: Monthly Revenue by Feature

derived_plot = derived.copy()
derived_plot["month"] = derived_plot["week"].dt.to_period("M").astype(str)
monthly_rev = derived_plot.groupby(["month", "feature"])["total_revenue"].sum().reset_index()

fig = px.bar(
    monthly_rev, x="month", y="total_revenue",
    color="feature", barmode="group",
    title="Monthly Revenue by Feature",
    template="plotly_white",
)
fig.update_layout(width=1100, height=500, xaxis_tickangle=-45)
fig.show()